In [ ]:
# =====================================================================
# ENVIRONMENT GUARD — run this cell first, before any TensorFlow import.
#
# This package requires Keras 2. Under Keras 3 the L1 activity regularizer
# on the autoencoder's latent layer drives it to zero: the model degenerates
# to predicting the per-feature mean and detects nothing. The failure is
# SILENT -- the healthy false-alarm rate still reads 2%.
#
# TensorFlow 2.16+ defaults to Keras 3, so tf-keras must be installed and
# TF_USE_LEGACY_KERAS set before TensorFlow is imported.
#
# See REPRODUCIBILITY.md, Section 0.
# =====================================================================

import os
import sys

os.environ["TF_USE_LEGACY_KERAS"] = "1"

# Make the modules in code/ importable regardless of the working directory.
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")) or ".")

import tensorflow as tf

_keras_version = getattr(tf.keras, "__version__", None)
if _keras_version is None:
    import tf_keras
    _keras_version = tf_keras.__version__

print("TensorFlow", tf.__version__)
print("Keras     ", _keras_version)

if not str(_keras_version).startswith("2."):
    raise RuntimeError(
        f"Keras {_keras_version} is active, but this package requires Keras 2.\n"
        "Install it and restart the kernel:\n"
        "    pip install tf-keras\n"
        "Then re-run this cell before anything else.\n"
        "See REPRODUCIBILITY.md, Section 0."
    )

print("Keras 2 active - safe to proceed.")

In [11]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, Dict, Protocol, Any
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.covariance import EllipticEnvelope
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

In [13]:
# Dataset location is resolved by data_paths.py:
#   1. $CWRU_DATA_ROOT   2. <deposit>/data/CWRU   3. ../CWRU_Bearing_NumPy-main/Data
# See data/README.md. Raw data is not redistributed with this deposit.
from data_paths import cwru_root

data_root = cwru_root()
print("CWRU data root:", data_root)

In [14]:
def load_cwru_signal(bearing_id, regime):
    bearing_rpm = f"{bearing_id} RPM"
    folder = data_root / bearing_rpm

    if regime == "healthy":
        d = np.load(folder / f"{bearing_id}_Normal.npz")
        return d["DE"]

    patterns = {
        "ball":  f"{bearing_id}_B_*_DE12.npz",
        "inner": f"{bearing_id}_IR_*_DE12.npz",
        "outer": f"{bearing_id}_OR*@*_DE12.npz",
    }

    signals = []
    for f in sorted(folder.glob(patterns[regime])):
        d = np.load(f)
        signals.append(d["DE"])

    return np.concatenate(signals)

In [15]:
# ---------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------

GLOBAL_SEED = 42


def set_all_seeds(seed: int = GLOBAL_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

In [31]:
def make_windows(
    x: np.ndarray,
    *,
    win: int = 1200,
    step: int = 1200,
) -> np.ndarray:
    """
    Split a 1-D signal into fixed-length chronological windows.

    Windows are identical to those used in the Conv1D Autoencoder
    experiments so every detector is evaluated on exactly the same
    samples.
    """
    x = np.asarray(x, dtype=np.float32).ravel()

    n = (len(x) - win) // step + 1

    if n <= 0:
        raise ValueError(
            f"Signal length ({len(x)}) is smaller than window size ({win})."
        )

    return np.stack(
        [x[i * step : i * step + win] for i in range(n)],
        axis=0,
    )

In [32]:
# ---------------------------------------------------------------------
# Common detector interface
# ---------------------------------------------------------------------

class AnomalyDetector(Protocol):
    """Minimal interface required by the generic experiment runner."""

    def fit(self, X_train: np.ndarray, X_val: np.ndarray | None = None) -> "AnomalyDetector":
        ...

    def score(self, X: np.ndarray) -> np.ndarray:
        """Return one anomaly score per sample; larger means more anomalous."""
        ...


In [33]:
# ---------------------------------------------------------------------
# Dataset definition
# ---------------------------------------------------------------------

@dataclass(frozen=True)
class HealthySplit:
    train: np.ndarray
    calibration: np.ndarray
    test: np.ndarray


@dataclass(frozen=True)
class CWRUDataset:
    bearing_id: str
    healthy: HealthySplit
    faults: Dict[str, np.ndarray]
    scaler: StandardScaler


def chronological_split(
    X: np.ndarray,
    train_fraction: float = 0.60,
    calibration_fraction: float = 0.20,
) -> HealthySplit:
    """
    Chronological split avoids overlapping or near-duplicate windows leaking
    between training, threshold calibration, and healthy evaluation.
    """
    n = len(X)
    n_train = int(np.floor(n * train_fraction))
    n_cal = int(np.floor(n * calibration_fraction))

    if n_train < 2 or n_cal < 2 or (n - n_train - n_cal) < 2:
        raise ValueError(
            f"Not enough healthy windows ({n}) for train/calibration/test split."
        )

    return HealthySplit(
        train=X[:n_train],
        calibration=X[n_train:n_train + n_cal],
        test=X[n_train + n_cal:],
    )


def build_cwru_dataset(
    bearing_id: str,
    *,
    win: int = 1200,
    step: int = 1200,
    train_fraction: float = 0.60,
    calibration_fraction: float = 0.20,
) -> CWRUDataset:
    """
    Build one asset-specific CWRU dataset.

    The same windows and splits are reused for every detector.

    Protocol
    --------
    - Healthy observations are split chronologically into:
        train / calibration / test.
    - The scaler is fitted only on healthy training windows.
    - The anomaly threshold is calibrated only on held-out healthy
      calibration windows.
    - Healthy false-alarm rate is measured on a separate healthy test set.
    - Fault observations are never used for training, scaling, or calibration.
    """

    signals = {
        "healthy": load_cwru_signal(bearing_id, "healthy"),
        "ball": load_cwru_signal(bearing_id, "ball"),
        "inner_race": load_cwru_signal(bearing_id, "inner"),
        "outer_race": load_cwru_signal(bearing_id, "outer"),
    }

    raw_windows = {}

    for regime, signal in signals.items():
        signal = np.asarray(signal, dtype=np.float32).squeeze()

        if signal.ndim != 1:
            raise ValueError(
                f"Expected a 1-D signal for {bearing_id=} and {regime=}, "
                f"but received shape {signal.shape}."
            )

        windows = make_windows(
            signal,
            win=win,
            step=step,
        ).astype(np.float32)

        if len(windows) == 0:
            raise ValueError(
                f"No windows generated for {bearing_id=} and {regime=}. "
                f"Signal length={len(signal)}, win={win}, step={step}."
            )

        raw_windows[regime] = windows

    healthy_raw_split = chronological_split(
        raw_windows["healthy"],
        train_fraction=train_fraction,
        calibration_fraction=calibration_fraction,
    )

    # Fit preprocessing exclusively on real healthy training data.
    scaler = StandardScaler()
    scaler.fit(healthy_raw_split.train)

    healthy = HealthySplit(
        train=scaler.transform(
            healthy_raw_split.train
        ).astype(np.float32),

        calibration=scaler.transform(
            healthy_raw_split.calibration
        ).astype(np.float32),

        test=scaler.transform(
            healthy_raw_split.test
        ).astype(np.float32),
    )

    faults = {
        regime: scaler.transform(windows).astype(np.float32)
        for regime, windows in raw_windows.items()
        if regime != "healthy"
    }

    return CWRUDataset(
        bearing_id=bearing_id,
        healthy=healthy,
        faults=faults,
        scaler=scaler,
    )

In [34]:
# ---------------------------------------------------------------------
# Detector implementations
# ---------------------------------------------------------------------

class Conv1DAutoencoderDetector:
    """
    Same Conv1D architecture as the original notebook.

    This is the ideal asset-specific baseline:
    one AE is trained using real healthy data from each CWRU operating condition.
    """

    def __init__(
        self,
        *,
        latent_dim: int = 16,
        learning_rate: float = 1e-3,
        batch_size: int = 32,
        epochs: int = 100,
        verbose: int = 0,
        seed: int = GLOBAL_SEED,
    ) -> None:
        self.latent_dim = latent_dim
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epochs = epochs
        self.verbose = verbose
        self.seed = seed
        self.model: Model | None = None

    @staticmethod
    def _as_3d(X: np.ndarray) -> np.ndarray:
        if X.ndim != 2:
            raise ValueError(f"Expected 2-D windows, got shape {X.shape}.")
        return X[..., np.newaxis].astype(np.float32)

    def _build(self, sig_len: int) -> Model:
        inp = keras.Input(shape=(sig_len, 1), name="signal_in")

        x = layers.Conv1D(
            32, kernel_size=16, strides=2, padding="same", activation="relu"
        )(inp)
        x = layers.Conv1D(
            64, kernel_size=8, strides=2, padding="same", activation="relu"
        )(x)
        x = layers.Conv1D(
            128, kernel_size=4, strides=2, padding="same", activation="relu"
        )(x)

        conv_shape = tuple(int(v) for v in x.shape[1:])
        x = layers.Flatten()(x)
        latent = layers.Dense(
            self.latent_dim,
            activity_regularizer=keras.regularizers.l1(1.5e-4),
            name="latent",
        )(x)

        y = layers.Dense(conv_shape[0] * conv_shape[1], activation="relu")(latent)
        y = layers.Reshape(conv_shape)(y)
        y = layers.Conv1DTranspose(
            128, kernel_size=4, strides=2, padding="same", activation="relu"
        )(y)
        y = layers.Conv1DTranspose(
            64, kernel_size=8, strides=2, padding="same", activation="relu"
        )(y)
        y = layers.Conv1DTranspose(
            32, kernel_size=16, strides=2, padding="same", activation="relu"
        )(y)
        y = layers.Conv1D(
            1, kernel_size=1, padding="same", activation="linear", name="signal_out"
        )(y)

        out_len = int(y.shape[1])
        if out_len > sig_len:
            y = layers.Cropping1D((0, out_len - sig_len))(y)
        elif out_len < sig_len:
            y = layers.ZeroPadding1D((0, sig_len - out_len))(y)

        model = Model(inp, y, name="asset_specific_autoencoder")
        model.compile(
            optimizer=keras.optimizers.Adam(self.learning_rate),
            loss="mse",
        )
        return model

    def fit(
        self,
        X_train: np.ndarray,
        X_val: np.ndarray | None = None,
    ) -> "Conv1DAutoencoderDetector":
        set_all_seeds(self.seed)

        X_train_3d = self._as_3d(X_train)
        X_val_3d = self._as_3d(X_val) if X_val is not None else None

        self.model = self._build(X_train.shape[1])

        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor="val_loss" if X_val is not None else "loss",
                patience=10,
                restore_best_weights=True,
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss" if X_val is not None else "loss",
                factor=0.5,
                patience=5,
                min_lr=1e-5,
            ),
        ]

        validation_data = (
            (X_val_3d, X_val_3d) if X_val_3d is not None else None
        )

        self.model.fit(
            X_train_3d,
            X_train_3d,
            validation_data=validation_data,
            epochs=self.epochs,
            batch_size=self.batch_size,
            callbacks=callbacks,
            verbose=self.verbose,
        )
        return self

    def score(self, X: np.ndarray) -> np.ndarray:
        if self.model is None:
            raise RuntimeError("Detector must be fitted before scoring.")

        X_3d = self._as_3d(X)
        X_rec = self.model.predict(X_3d, verbose=0)
        return np.mean((X_3d - X_rec) ** 2, axis=(1, 2))


In [35]:
class OneClassSVMDetector:
    """
    Asset-specific OC-SVM trained on real healthy windows.

    score_samples gives larger values for normal points, so we negate it to
    preserve the common convention: larger score = more anomalous.
    """

    def __init__(
        self,
        *,
        nu: float = 0.02,
        gamma: str | float = "scale",
        pca_components: int | None = None,
    ) -> None:
        steps: list[tuple[str, Any]] = []
        if pca_components is not None:
            steps.append(
                (
                    "pca",
                    PCA(
                        n_components=pca_components,
                        random_state=GLOBAL_SEED,
                    ),
                )
            )
        steps.append(("ocsvm", OneClassSVM(kernel="rbf", nu=nu, gamma=gamma)))
        self.model = Pipeline(steps)

    def fit(
        self,
        X_train: np.ndarray,
        X_val: np.ndarray | None = None,
    ) -> "OneClassSVMDetector":
        self.model.fit(X_train)
        return self

    def score(self, X: np.ndarray) -> np.ndarray:
        return -self.model.score_samples(X)


class EllipticEnvelopeDetector:
    """
    Robust Gaussian baseline.

    PCA is strongly recommended because raw 1200-D covariance estimation is
    ill-conditioned when the number of healthy windows is limited.
    """

    def __init__(
        self,
        *,
        contamination: float = 0.02,
        pca_components: int = 16,
        support_fraction: float | None = None,
    ) -> None:
        self.model = Pipeline(
            [
                (
                    "pca",
                    PCA(
                        n_components=pca_components,
                        random_state=GLOBAL_SEED,
                    ),
                ),
                (
                    "elliptic",
                    EllipticEnvelope(
                        contamination=contamination,
                        support_fraction=support_fraction,
                        random_state=GLOBAL_SEED,
                    ),
                ),
            ]
        )

    def fit(
        self,
        X_train: np.ndarray,
        X_val: np.ndarray | None = None,
    ) -> "EllipticEnvelopeDetector":
        self.model.fit(X_train)
        return self

    def score(self, X: np.ndarray) -> np.ndarray:
        # score_samples: larger = more normal; negate for anomaly convention.
        return -self.model.score_samples(X)



In [36]:
# ---------------------------------------------------------------------
# Generic evaluation
# ---------------------------------------------------------------------

@dataclass(frozen=True)
class ExperimentResult:
    bearing_id: str
    detector: str
    threshold_percentile: float
    threshold: float
    healthy_far: float
    ball_dr: float
    inner_race_dr: float
    outer_race_dr: float
    n_train: int
    n_calibration: int
    n_healthy_test: int


def evaluate_detector(
    detector: AnomalyDetector,
    dataset: CWRUDataset,
    *,
    detector_name: str,
    threshold_percentile: float = 98.0,
) -> ExperimentResult:
    """
    Fair ideal-per-asset protocol.

    Training:
        healthy.train
    Model-selection / AE early stopping:
        healthy.calibration
    Threshold:
        percentile of detector scores on healthy.calibration
    Final FAR:
        healthy.test only
    Final DR:
        all fault windows
    """
    detector.fit(dataset.healthy.train, dataset.healthy.calibration)

    calibration_scores = detector.score(dataset.healthy.calibration)
    tau = float(np.percentile(calibration_scores, threshold_percentile))

    healthy_scores = detector.score(dataset.healthy.test)
    fault_scores = {
        name: detector.score(X)
        for name, X in dataset.faults.items()
    }

    return ExperimentResult(
        bearing_id=dataset.bearing_id,
        detector=detector_name,
        threshold_percentile=threshold_percentile,
        threshold=tau,
        healthy_far=float(np.mean(healthy_scores > tau)),
        ball_dr=float(np.mean(fault_scores["ball"] > tau)),
        inner_race_dr=float(np.mean(fault_scores["inner_race"] > tau)),
        outer_race_dr=float(np.mean(fault_scores["outer_race"] > tau)),
        n_train=len(dataset.healthy.train),
        n_calibration=len(dataset.healthy.calibration),
        n_healthy_test=len(dataset.healthy.test),
    )


def run_asset_specific_suite(
    model_factories: Dict[str, Callable[[], AnomalyDetector]],
    *,
    bearing_ids: tuple[str, ...] = ("1730", "1750", "1772", "1797"),
    win: int = 1200,
    step: int = 1200,
    threshold_percentile: float = 98.0,
) -> pd.DataFrame:
    """
    Train every model independently for every CWRU operating condition.

    The dataset is built once per bearing_id and reused unchanged for all
    detector factories, guaranteeing direct comparability.
    """
    rows: list[dict[str, Any]] = []

    for bearing_id in bearing_ids:
        dataset = build_cwru_dataset(
            bearing_id,
            win=win,
            step=step,
        )

        print(
            f"\n[{bearing_id} RPM] "
            f"train={len(dataset.healthy.train)}, "
            f"cal={len(dataset.healthy.calibration)}, "
            f"healthy_test={len(dataset.healthy.test)}"
        )

        for model_name, factory in model_factories.items():
            print(f"  Training {model_name}...")
            detector = factory()
            result = evaluate_detector(
                detector,
                dataset,
                detector_name=model_name,
                threshold_percentile=threshold_percentile,
            )
            rows.append(result.__dict__)

    df = pd.DataFrame(rows)

    metric_cols = [
        "healthy_far",
        "ball_dr",
        "inner_race_dr",
        "outer_race_dr",
    ]
    df[metric_cols] = 100.0 * df[metric_cols]
    return df


In [37]:
# ---------------------------------------------------------------------
# Generic evaluation
# ---------------------------------------------------------------------

@dataclass(frozen=True)
class ExperimentResult:
    bearing_id: str
    detector: str
    threshold_percentile: float
    threshold: float
    healthy_far: float
    ball_dr: float
    inner_race_dr: float
    outer_race_dr: float
    n_train: int
    n_calibration: int
    n_healthy_test: int


def evaluate_detector(
    detector: AnomalyDetector,
    dataset: CWRUDataset,
    *,
    detector_name: str,
    threshold_percentile: float = 98.0,
) -> ExperimentResult:
    """
    Fair ideal-per-asset protocol.

    Training:
        healthy.train
    Model-selection / AE early stopping:
        healthy.calibration
    Threshold:
        percentile of detector scores on healthy.calibration
    Final FAR:
        healthy.test only
    Final DR:
        all fault windows
    """
    detector.fit(dataset.healthy.train, dataset.healthy.calibration)

    calibration_scores = detector.score(dataset.healthy.calibration)
    tau = float(np.percentile(calibration_scores, threshold_percentile))

    healthy_scores = detector.score(dataset.healthy.test)
    fault_scores = {
        name: detector.score(X)
        for name, X in dataset.faults.items()
    }

    return ExperimentResult(
        bearing_id=dataset.bearing_id,
        detector=detector_name,
        threshold_percentile=threshold_percentile,
        threshold=tau,
        healthy_far=float(np.mean(healthy_scores > tau)),
        ball_dr=float(np.mean(fault_scores["ball"] > tau)),
        inner_race_dr=float(np.mean(fault_scores["inner_race"] > tau)),
        outer_race_dr=float(np.mean(fault_scores["outer_race"] > tau)),
        n_train=len(dataset.healthy.train),
        n_calibration=len(dataset.healthy.calibration),
        n_healthy_test=len(dataset.healthy.test),
    )


def run_asset_specific_suite(
    model_factories: Dict[str, Callable[[], AnomalyDetector]],
    *,
    bearing_ids: tuple[str, ...] = ("1730", "1750", "1772", "1797"),
    win: int = 1200,
    step: int = 1200,
    threshold_percentile: float = 98.0,
) -> pd.DataFrame:
    """
    Train every model independently for every CWRU operating condition.

    The dataset is built once per bearing_id and reused unchanged for all
    detector factories, guaranteeing direct comparability.
    """
    rows: list[dict[str, Any]] = []

    for bearing_id in bearing_ids:
        dataset = build_cwru_dataset(
            bearing_id,
            win=win,
            step=step,
        )

        print(
            f"\n[{bearing_id} RPM] "
            f"train={len(dataset.healthy.train)}, "
            f"cal={len(dataset.healthy.calibration)}, "
            f"healthy_test={len(dataset.healthy.test)}"
        )

        for model_name, factory in model_factories.items():
            print(f"  Training {model_name}...")
            detector = factory()
            result = evaluate_detector(
                detector,
                dataset,
                detector_name=model_name,
                threshold_percentile=threshold_percentile,
            )
            rows.append(result.__dict__)

    df = pd.DataFrame(rows)

    metric_cols = [
        "healthy_far",
        "ball_dr",
        "inner_race_dr",
        "outer_race_dr",
    ]
    df[metric_cols] = 100.0 * df[metric_cols]
    return df


# ---------------------------------------------------------------------
# Example experiment configuration
# ---------------------------------------------------------------------

MODEL_FACTORIES: Dict[str, Callable[[], AnomalyDetector]] = {
    "Asset-specific AE": lambda: Conv1DAutoencoderDetector(
        latent_dim=16,
        epochs=100,
        batch_size=32,
        verbose=0,
    ),
    "Asset-specific OC-SVM": lambda: OneClassSVMDetector(
        nu=0.02,
        gamma="scale",
        pca_components=32,
    ),
    "Asset-specific Elliptic Envelope": lambda: EllipticEnvelopeDetector(
        contamination=0.02,
        pca_components=16,
    ),
}

In [38]:
ideal_results = run_asset_specific_suite(
     MODEL_FACTORIES,
     bearing_ids=("1730", "1750", "1772", "1797"),
     win=1200,
     step=1200,
     threshold_percentile=98.0,
 )

display(ideal_results)


[1730 RPM] train=242, cal=80, healthy_test=82
  Training Asset-specific AE...


2026-07-15 19:56:02.983290: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2 Pro
2026-07-15 19:56:02.983833: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-07-15 19:56:02.983854: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-07-15 19:56:02.984458: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-07-15 19:56:02.984979: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-07-15 19:56:04.217876: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


  Training Asset-specific OC-SVM...
  Training Asset-specific Elliptic Envelope...

[1750 RPM] train=242, cal=80, healthy_test=82
  Training Asset-specific AE...
  Training Asset-specific OC-SVM...
  Training Asset-specific Elliptic Envelope...

[1772 RPM] train=241, cal=80, healthy_test=82
  Training Asset-specific AE...
  Training Asset-specific OC-SVM...
  Training Asset-specific Elliptic Envelope...

[1797 RPM] train=121, cal=40, healthy_test=42
  Training Asset-specific AE...
  Training Asset-specific OC-SVM...
  Training Asset-specific Elliptic Envelope...


,bearing_id,detector,threshold_percentile,threshold,healthy_far,ball_dr,inner_race_dr,outer_race_dr,n_train,n_calibration,n_healthy_test
0,1730,Asset-specific AE,98.0,1.312155,15.853659,100.000000,100.000000,100.000000,242,80,82
1,1730,Asset-specific OC-SVM,98.0,-0.566257,2.439024,1.975309,25.123153,5.477528,242,80,82
2,1730,Asset-specific Elliptic Envelope,98.0,19.198287,1.219512,0.987654,25.862069,6.179775,242,80,82
3,1750,Asset-specific AE,98.0,0.714190,6.097561,100.000000,100.000000,100.000000,242,80,82
4,1750,Asset-specific OC-SVM,98.0,-0.676011,6.097561,18.518519,25.185185,5.907173,242,80,82
5,1750,Asset-specific Elliptic Envelope,98.0,17.595343,14.634146,2.469136,22.716049,0.000000,242,80,82
6,1772,Asset-specific AE,98.0,1.109930,31.707317,100.000000,100.000000,100.000000,241,80,82
7,1772,Asset-specific OC-SVM,98.0,-0.441966,18.292683,0.987654,24.938272,0.000000,241,80,82
8,1772,Asset-specific Elliptic Envelope,98.0,111.832192,6.097561,0.000000,0.000000,0.000000,241,80,82
9,1797,Asset-specific AE,98.0,0.521162,0.000000,100.000000,100.000000,100.000000,121,40,42


In [ ]:
summary = (
    ideal_results
    .groupby("detector")[
        ["healthy_far", "ball_dr", "inner_race_dr", "outer_race_dr"]
    ]
    .agg(["mean", "std"])
    .round(2)
)
display(summary)
